# OpenVSP: size the plane with OpenVSP parasite drag

`run_pipeline` sizes the plane with `evaluate`, gives the wing (area, aspect ratio, taper) to the OpenVSP model, runs the parasite drag build-up and feeds the result back into `evaluate`, until the plane's $C_{D,0}$ stops changing.

The drag is fed back in two parts, so `evaluate` gets $C_{D,0}$ right for any wing area $S$ it tries:

- `c_d_0_wing`: the wing's own $C_D$ (referenced to its own area, almost independent of size)
- `f_fixed`: drag area $C_D \cdot S$ of fuselage and tails [m²], which don't change with the wing

$$C_{D,0}(S) = c_{d,0,	ext{wing}} + f_	ext{fixed} / S$$

The `.vsp3` file is set in `ardea_sizer/pipeline/_open_vsp.py` and stays unchanged on disk unless `SAVE_AS` is set. Start values are `c_d_0_wing` and `f_fixed` in `base.json`.

In [7]:
from pathlib import Path

import pandas as pd
import openvsp as vsp

import ardea_sizer as az
from ardea_sizer.pipeline._open_vsp import VSP3_FILE, WING_NAME
from ardea_sizer.pipeline._pipeline import run_pipeline

BASE_FILE = "base.json"
SAVE_AS = None                # e.g. "models/20261012_Convent_BM_V3_sized.vsp3"

In [8]:
def wing_summary() -> pd.Series:
    wing_id = vsp.FindGeomsWithName(WING_NAME)[0]
    g = lambda name, group: vsp.GetParmVal(wing_id, name, group)
    return pd.Series({
        "area [m²]": g("TotalArea", "WingGeom"),
        "aspect ratio": g("TotalAR", "WingGeom"),
        "span [m]": g("TotalSpan", "WingGeom"),
        "mean chord [m]": g("TotalChord", "WingGeom"),
        "MAC [m]": g("MAC", "WingGeom"),
        "root chord [m]": g("Root_Chord", "XSec_1"),
        "tip chord [m]": g("Tip_Chord", "XSec_1"),
        "taper": g("Taper", "XSec_1"),
    })


def drag_components() -> pd.DataFrame:
    # last parasite drag run (the one inside optimize)
    rid = vsp.FindLatestResultsID("Parasite_Drag")
    return pd.DataFrame({
        "S_wet [m²]": vsp.GetDoubleResults(rid, "Comp_Swet", 0),
        "f [m²]": vsp.GetDoubleResults(rid, "Comp_f", 0),
        "CD": vsp.GetDoubleResults(rid, "Comp_CD", 0),
    }, index=vsp.GetStringResults(rid, "Comp_Label", 0))

## Wing in the file (before sizing)

In [9]:
vsp.ClearVSPModel()
vsp.ReadVSPFile(str(VSP3_FILE))
vsp.Update()
wing_file = wing_summary()
wing_file.round(4).to_frame(VSP3_FILE.name)

,20261012_Convent_BM_V3.vsp3
area [m²],0.4000
aspect ratio,8.0000
span [m],1.7889
mean chord [m],0.2236
MAC [m],0.2343
root chord [m],0.3084
tip chord [m],0.1388
taper,0.4500


## Sizing loop with drag feedback

In [10]:
base_start = az.load(BASE_FILE)
plane, base = run_pipeline(base_start)

pd.Series({
    "c_d_0_wing start": base_start.assumptions.c_d_0_wing,
    "c_d_0_wing converged": base.assumptions.c_d_0_wing,
    "f_fixed start [m²]": base_start.assumptions.f_fixed,
    "f_fixed converged [m²]": base.assumptions.f_fixed,
    "CD0 of the plane": float(plane.c_d_0),
    "fuel mass [kg]": float(plane.fuel_mass),
    "wing area [m²]": float(plane.wing_area),
    "aspect ratio": float(plane.aspect_ratio),
    "taper": float(plane.taper),
    "range [km]": float(plane.range) / 1000,
    "power required [kW]": float(plane.power_req) / 1000,
    "power available [kW]": float(plane.power_available),
}).round(4).to_frame("sized plane")

Run 1: c_d_0_wing 0.00800 -> 0.00804   f_fixed 0.00400 -> 0.00382 m^2   CD0 0.01691 -> 0.01655   S 0.4489 m^2   fuel 5.590 kg
Run 2: c_d_0_wing 0.00804 -> 0.00803   f_fixed 0.00382 -> 0.00382 m^2   CD0 0.01660 -> 0.01660   S 0.4459 m^2   fuel 5.458 kg


,sized plane
c_d_0_wing start,0.0080
c_d_0_wing converged,0.0080
f_fixed start [m²],0.0040
f_fixed converged [m²],0.0038
CD0 of the plane,0.0166
fuel mass [kg],5.4585
wing area [m²],0.4459
aspect ratio,5.0000
taper,0.4500
range [km],1050.5798


## Sized wing in OpenVSP

The model in memory now holds the wing of the last run. Parasite drag per component ($C_D$ referenced to the sized wing area):

In [11]:
display(drag_components().round(5))

wing_sized = wing_summary()
comparison = pd.DataFrame({"file": wing_file, "sized": wing_sized})
comparison["change [%]"] = 100 * (comparison["sized"] / comparison["file"] - 1)
comparison.round(4)

,S_wet [m²],f [m²],CD
ARDEA_D3_Stack,0.76722,0.00268,0.00602
vertical stabilizer,0.06139,0.00039,0.00087
Horizontal stabilizer,0.11930,0.00075,0.00168
Main Wing,0.71639,0.00358,0.00803


,file,sized,change [%]
area [m²],0.4000,0.4459,11.4812
aspect ratio,8.0000,5.0000,-37.5000
span [m],1.7889,1.4932,-16.5280
mean chord [m],0.2236,0.2986,33.5552
MAC [m],0.2343,0.3130,33.5552
root chord [m],0.3084,0.4119,33.5552
tip chord [m],0.1388,0.1854,33.5552
taper,0.4500,0.4500,0.0000


In [12]:
if SAVE_AS:
    vsp.WriteVSPFile(str(Path(SAVE_AS).resolve()))
    print(f"Saved to {SAVE_AS}")